<a href="https://colab.research.google.com/github/rajilsaj/nasa-mosaics-project/blob/xgboost/notebooks/03_negative_sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!ls "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows"


train_windows.csv


In [7]:
import pandas as pd
import numpy as np
import os

SPLIT_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/splits"
WINDOW_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows"
OUTPUT_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows"

WINDOW_SIZE = 60
BUFFER = 50
NEG_RATIO = 1.0   # 1:1 for training

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [8]:
split_name = "train"  # change to val or test later

ml_df = pd.read_csv(f"{SPLIT_DIR}/ml_{split_name}.csv")
positive_df = pd.read_csv(f"{WINDOW_DIR}/{split_name}_windows.csv")

print("ML samples:", len(ml_df))
print("Positive windows:", positive_df["window_id"].nunique())


ML samples: 2513117
Positive windows: 225


In [9]:
forbidden = np.zeros(len(ml_df), dtype=bool)

# 1️⃣ Mark precursor zones
forbidden[ml_df["gt_detection_win"] == True] = True

# 2️⃣ Add buffer around each positive window
for window_id in positive_df["window_id"].unique():

    window_data = positive_df[positive_df["window_id"] == window_id]

    start_sclk = window_data["SCLK"].iloc[0]
    end_sclk = window_data["SCLK"].iloc[-1]

    start_idx = ml_df.index[ml_df["SCLK"] == start_sclk][0]
    end_idx = ml_df.index[ml_df["SCLK"] == end_sclk][0]

    buffer_start = max(0, start_idx - BUFFER)
    buffer_end = min(len(ml_df), end_idx + BUFFER)

    forbidden[buffer_start:buffer_end] = True

print("Forbidden samples:", forbidden.sum())


Forbidden samples: 13455


In [10]:
valid_starts = []

for i in range(len(ml_df) - WINDOW_SIZE):
    if not forbidden[i:i + WINDOW_SIZE].any():
        valid_starts.append(i)

print("Valid negative starting positions:", len(valid_starts))


Valid negative starting positions: 2486807


In [11]:
num_positive = positive_df["window_id"].nunique()
num_negative = int(num_positive * NEG_RATIO)

rng = np.random.default_rng(42)
sampled_starts = rng.choice(valid_starts, size=num_negative, replace=False)

negative_windows = []

for window_id, start_idx in enumerate(sampled_starts):

    window = ml_df.iloc[start_idx:start_idx + WINDOW_SIZE].copy()

    window["window_id"] = window_id
    window["label"] = 0
    window["event_sclk"] = -1

    negative_windows.append(window)

print("Negative windows created:", len(negative_windows))


Negative windows created: 225
